# Part 1 : Cortex AI 関数

Cortex AI の詳細は別のスライドで解説します。
Part1 では、**SQL から直接呼び出せる Cortex AI 関数を使って、テキストや画像から構造化データを作る**体験を行います。

本パートで使う関数は以下の 4 つです。

| 関数 | できること |
|---|---|
| **AI_EXTRACT** | テキストや PDF から指定した項目を抽出し、JSON で返す |
| **AI_CLASSIFY** | テキストをあらかじめ定義したカテゴリに分類する |
| **AI_SIMILARITY** | 2 つのテキストの意味的な類似度（0〜1）を返す |
| **AI_COMPLETE** | プロンプトを渡して自由なテキスト生成・画像分析を行う |

In [ ]:
-- コンテキスト設定
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE AI_HANDSON_WH;
USE DATABASE AI_HANDSON_DB;
USE SCHEMA ANALYTICS;


---
## 1. ウェアハウスの作成

この後のハンズオンで利用するウェアハウスとして、アダプティブウェアハウスを作成します。
この後の手順ではSQLで作成する流れになっていますが、GUIでも作成可能です。

合わせて確認してみてください。

> **GUI での作成方法**
> 
> Snowsight 左メニュー **コンピュート → ウェアハウス → + ウェアハウス** で、タイプ に **適応型** を選択します。
> ※ACCOUNT_ADMINなどウェアハウス作成権限があるロールを選択して行ってください

### SQL で作成する

In [ ]:
%%sql -r dataframe_1_2
-- アダプティブウェアハウスを作成
CREATE OR REPLACE ADAPTIVE WAREHOUSE AI_HANDSON_ADAPTIVE_WH
    MAX_QUERY_PERFORMANCE_LEVEL = MEDIUM
    QUERY_THROUGHPUT_MULTIPLIER = 2
    COMMENT = 'Part 1 アダプティブウェアハウス ハンズオン用';

In [ ]:
%%sql -r dataframe_1_3
-- 作成結果を確認
-- 確認ポイント:
--   1. "type" = ADAPTIVE → 従来の STANDARD でなくアダプティブウェアハウスとして作成されている
--   2. "size" = NULL → アダプティブWHにはサイズ（XS/S/M/...）の概念がなく、ワークロードに応じて自動スケール
--   3. "state" = ENABLED → アダプティブWHは SUSPEND/RESUME ではなく ENABLE/DISABLE で制御する（STARTED/SUSPENDED にはならない）
--   4. "max_query_performance_level" = MEDIUM → 1クエリあたりに使えるコンピュートの上限（XSMALL〜X4LARGE）
--   5. "query_throughput_multiplier" = 2 → 同時実行性能の倍率（デフォルト2、大きいほど並列処理に強い。0=無制限）
SHOW WAREHOUSES LIKE 'AI_HANDSON_ADAPTIVE_WH';

In [ ]:
%%sql -r dataframe_1_4
-- アダプティブウェアハウスでクエリを実行してみる
USE WAREHOUSE AI_HANDSON_ADAPTIVE_WH;

-- 商品別の売上を集計
SELECT
    p.product_name,
    p.category,
    SUM(s.units_sold)   AS total_units,
    SUM(s.sales_amount) AS total_sales
FROM daily_sales s
JOIN products p ON s.product_id = p.product_id
GROUP BY p.product_name, p.category
ORDER BY total_sales DESC;

---
## 2. Cortex AI 関数

Cortex AI 関数を使って、テキストや画像から構造化データを作ります。

### AI_EXTRACT — 構造化データを取り出す

![AI_EXTRACT関数](data/slide/AI_EXTRACT.png)

サンプルデータとして、**SNS に投稿されたテキスト（`sns_mentions` テーブル）** と **商品スペックシートの PDF** を用意しています。
これらの非構造化データから、商品名・カテゴリ・問い合わせタイプなどの項目を `AI_EXTRACT` で抜き出して試していきます。


In [ ]:
%%sql -r dataframe_1_5
-- 元データの確認(SNS 投稿のサンプルデータ)
SELECT
    post_id,
    platform,
    username,
    content,
    likes,
    posted_at
FROM sns_mentions
ORDER BY posted_at
LIMIT 10;

In [ ]:
%%sql -r dataframe_1_6
-- AI_EXTRACT を 5 件だけ試してみる
-- 投稿文から「商品名」「カテゴリ」「問い合わせタイプ」の3項目を抽出する
SELECT
    content,
    AI_EXTRACT(
        content,
        {'product_name': '投稿で言及されている商品名',
         'category':     '商品カテゴリ(ファッション / インテリア / テック など)',
         'inquiry_type': '投稿の種類(質問 / レビュー / クレーム / 称賛 / 提案)'}
    ) AS extracted
FROM sns_mentions
LIMIT 5;

`AI_EXTRACT` の結果は **JSON（VARIANT 型）** でそのまま返ってきます。
実務ではこの JSON を**列に展開**して、通常のテーブルとして扱えるようにします。

展開には `:response.<項目名>` という半構造化データのアクセス記法を使います。

```
extracted:response.product_name::VARCHAR AS product_name
```

- `extracted` … `AI_EXTRACT` の戻り値（VARIANT 型）
- `:response` … JSON 内の `response` キーにアクセス
- `.product_name` … その中の `product_name` を取り出す
- `::VARCHAR` … 文字列型にキャスト

In [ ]:
%%sql -r dataframe_1_7
-- 抽出結果を列に展開する
SELECT
    content,
    extracted:response.product_name::VARCHAR AS product_name,
    extracted:response.category::VARCHAR     AS category,
    extracted:response.inquiry_type::VARCHAR AS inquiry_type
FROM (
    SELECT
        content,
        AI_EXTRACT(
            content,
            {'product_name': '投稿で言及されている商品名',
             'category':     '商品カテゴリ(ファッション / インテリア / テック など)',
             'inquiry_type': '投稿の種類(質問 / レビュー / クレーム / 称賛 / 提案)'}
        ) AS extracted
    FROM sns_mentions
);

#### PDF から抽出する

テキストと同じ要領で、ステージ上の **PDF** からも抽出できます。
第1引数を `TO_FILE('@ステージ', 'パス')` に変えるだけです。

`setup.sql` でロード済みの商品スペックシート 5 枚から情報を抜き出してみましょう。

In [ ]:
%%sql -r dataframe_1_8
-- ステージ上の商品スペック PDF を確認
SELECT relative_path, size
FROM DIRECTORY(@DATA_STAGE)
WHERE relative_path ILIKE '%.pdf'
ORDER BY relative_path;

In [ ]:
%%sql -r dataframe_1_9
-- PDF から商品情報を抽出する
SELECT
    relative_path AS pdf_file,
    extracted:response.product_name::VARCHAR AS product_name,
    extracted:response.category::VARCHAR     AS category,
    extracted:response.price::VARCHAR        AS price,
    extracted:response.material::VARCHAR     AS material
FROM (
    SELECT
        relative_path,
        AI_EXTRACT(
            TO_FILE('@DATA_STAGE', relative_path),
            {'product_name': '商品名',
             'category':     '商品カテゴリ',
             'price':        '価格（税込）',
             'material':     '素材・材質'}
        ) AS extracted
    FROM DIRECTORY(@DATA_STAGE)
    WHERE relative_path ILIKE '%.pdf'
);

### AI_CLASSIFY — テキストを分類する

![AI_CLASSIFY関数](data/slide/AI_CLASSIFY.png)

サンプルデータとして、先ほどと同じ **SNS 投稿テキスト（`sns_mentions` テーブル）** を使います。
各投稿を `['称賛', 'クレーム', '質問', '提案']` の 4 カテゴリに自動分類し、カテゴリ別の集計ができる状態にしていきます。


In [ ]:
%%sql -r dataframe_1_10
-- AI_CLASSIFY を 5 件だけ試してみる
SELECT
    content,
    AI_CLASSIFY(
        content,
        ['称賛', 'クレーム', '質問', '提案']
    ) AS classified
FROM sns_mentions
LIMIT 5;

In [ ]:
%%sql -r dataframe_1_11
-- 分類結果をテーブルに保存する

CREATE OR REPLACE TABLE sns_mentions_classified AS
SELECT
    post_id,
    platform,
    username,
    content,
    likes,
    posted_at,
    AI_CLASSIFY(
        content,
        ['称賛', 'クレーム', '質問', '提案']
    ):labels[0]::VARCHAR AS post_category
FROM sns_mentions;

In [ ]:
%%sql -r dataframe_1_12
-- 分類結果を確認
SELECT
    post_category,
    content,
    likes
FROM sns_mentions_classified
LIMIT 20;

In [ ]:
%%sql -r dataframe_1_13
-- 構造化されたので、普通の SQL で集計できる
-- カテゴリ別の投稿数・平均いいね数・構成比
SELECT
    post_category,
    COUNT(*)                                   AS post_count,
    ROUND(AVG(likes), 1)                       AS avg_likes,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM sns_mentions_classified
GROUP BY post_category
ORDER BY post_count DESC;

### AI_SIMILARITY — 類似度の算出

![AI_SIMILARITY関数](data/slide/AI_SIMILARITY.png)

サンプルデータとして、**仕入先から届いた商品リスト（`supplier_products_v2` テーブル）** と **自社の商品マスタ（`dim_products` テーブル）** を使います。
実務では「同じ商品なのに名前が微妙に違う」ケースが頻発しますが、完全一致 JOIN では拾えません。
`AI_SIMILARITY` で意味的な類似度を計算し、名寄せ候補を見つけていきます。

**スコアは 0〜1 の数値。高いほど意味が近く、1 が完全一致。**

**表記揺れの例**
- 全角/半角:`ＬＥＤデスクライト` → `LEDデスクライト`
- 同義語:`ライト` ↔ `ランプ` ↔ `灯`
- 語順逆転:`ウール 高品質ブランケット` → `高品質ウールブランケット`
- ブランド名付き:`GlacierStyle スタイリッシュデスクライト`

**処理の考え方:**
- 仕入先商品(50件)× 商品マスタ(576件)= 全 28,800 通りの組み合わせを計算
- 各仕入先商品に対して「最もスコアが高いマスタ商品」を候補として選ぶ
- スコアが高い = AI が「似ている」と判断した組み合わせ
  (最終的に「同じ商品か」の判断は人間が行う)


In [ ]:
%%sql -r dataframe_1_19
-- 仕入先データのプレビュー(表記揺れを含む)
SELECT
    supplier_product_id,
    supplier_product_name,
    supplier_name,
    supplier_price,
    supplier_category
FROM supplier_products_v2;

In [ ]:
%%sql -r dataframe_1_20
-- 商品マスタのプレビュー(突合先)
SELECT
    product_id,
    product_name,
    category_l2,
    brand,
    current_price
FROM dim_products;

In [ ]:
%%sql -r similarity_demo
-- 2つの例で AI_SIMILARITY のスコアを比較する
-- 例1: 全角/半角の違いだけ → ほぼ 1.0
-- 例2: 「壁に飾る写真入れ」と「ウォールフレーム」→ 文字は全然違うが意味は同じ
SELECT
    supplier_name,
    master_name,
    ROUND(AI_SIMILARITY(supplier_name, master_name), 3) AS similarity
FROM (
    SELECT 'ＬＥＤ調光デスクライト' AS supplier_name, 'LED調光デスクライト' AS master_name
    UNION ALL
    SELECT '壁に飾る写真入れ', 'ウォールフレーム'
);

In [ ]:
%%sql -r dataframe_1_21
-- AI_SIMILARITY による名寄せ(サンプル50件)
-- 少し時間がかかります(CROSS JOIN のため)
--
-- 処理の流れ:
--   1. 仕入先商品(50件)× 商品マスタ(576件)を全組み合わせで比較
--   2. 各仕入先商品に対して最もスコアが高いマスタ商品を選ぶ(rank = 1)
--   3. スコアが高いほど AI が「同じ商品」と判断した組み合わせ
CREATE OR REPLACE TABLE work_ai_similarity_match AS
WITH sample_suppliers AS (
    SELECT * FROM supplier_products_v2 SAMPLE (50 ROWS)
),
similarity_calc AS (
    SELECT
        sp.supplier_product_id,
        sp.supplier_product_name,
        dp.product_id,
        dp.product_name,
        AI_SIMILARITY(sp.supplier_product_name, dp.product_name) AS ai_sim,
        ROW_NUMBER() OVER (
            PARTITION BY sp.supplier_product_id
            ORDER BY AI_SIMILARITY(sp.supplier_product_name, dp.product_name) DESC
        ) AS rank
    FROM sample_suppliers sp
    CROSS JOIN dim_products dp
)
SELECT
    supplier_product_id,
    supplier_product_name,
    product_id   AS matched_product_id,
    product_name AS matched_product_name,
    ROUND(ai_sim, 3) AS ai_similarity
FROM similarity_calc
WHERE rank = 1;

-- 結果確認(スコアが高い順)
SELECT * FROM work_ai_similarity_match
ORDER BY ai_similarity DESC;

### AI_COMPLETE — 自由なプロンプトを実行

![AI_COMPLETE関数](data/slide/AI_COMPLETE.png)

AI_COMPLETEは画像や音声などの非構造化データも扱えます。

ここではサンプルデータとして、**インフルエンサーの投稿画像（`@POST_IMAGES` ステージ）** を使います。
`AI_COMPLETE` にプロンプトと画像を渡して、「人が写っているか」「どこで撮っているか」「商品を使っている状態か」などを構造化データにしていきます。


In [ ]:
%%sql -r dataframe_1_14
-- ステージ上の投稿画像を確認
SELECT
    relative_path,
    size
FROM DIRECTORY(@POST_IMAGES)
ORDER BY relative_path;

In [ ]:
%%sql -r dataframe_1_15
-- AI_COMPLETE で画像を 3 枚だけ分析してみる
-- JSON 形式で返すようにプロンプトで指示する
SELECT
    RELATIVE_PATH AS image_file,
    PARSE_JSON(
        REGEXP_REPLACE(
            AI_COMPLETE(
                'claude-sonnet-5',
                PROMPT(
                    $$以下の画像を分析して、JSON形式で結果を返してください。
JSONオブジェクトのみ返してください。説明文やコードブロック記法は不要です。

photo_main_subject: "商品" or "人物" or "両方"
has_person: true or false
location: "自宅" or "カフェ" or "屋外" or "スタジオ" or "空港" or "ホテル" or "電車内" or "その他"
product_usage: "未使用" or "使用中" or "ビフォーアフター"
color_tone: "暖色系" or "寒色系" or "パステル"

画像: {0}$$,
                    TO_FILE('@POST_IMAGES', RELATIVE_PATH)
                )
            ),
            '^[^{]*|[^}]*$', ''
        )
    ) AS ai_response
FROM DIRECTORY(@POST_IMAGES)
ORDER BY RELATIVE_PATH
LIMIT 3;

同じ処理を **全件**に対して実行し、テーブルに保存します。


In [ ]:
-- 全件を構造化して image_features テーブルを作成
CREATE OR REPLACE TABLE image_features AS
SELECT
    SPLIT_PART(RELATIVE_PATH, '/', -1) AS image_file,
    PARSE_JSON(
        REGEXP_REPLACE(
            AI_COMPLETE(
                'claude-sonnet-5',
                PROMPT(
                    $$以下の画像を分析して、JSON形式で結果を返してください。
JSONオブジェクトのみ返してください。説明文やコードブロック記法は不要です。

photo_main_subject: "商品" or "人物"
has_person: true or false
person_gender: "男性" or "女性" or null
person_size: "大きい" or "中" or "小さい" or null
expression: "笑顔" or "クール" or "自然体" or null
location: "自宅" or "カフェ" or "屋外" or "スタジオ" or "空港" or "ホテル" or "電車内" or "その他"
product_usage: "未使用" or "使用中" or "ビフォーアフター"
color_tone: "暖色系" or "寒色系" or "パステル"

画像: {0}$$,
                    TO_FILE('@POST_IMAGES', RELATIVE_PATH)
                )
            ),
            '^[^{]*|[^}]*$', ''
        )
    ) AS features_json,
    features_json:photo_main_subject::VARCHAR AS photo_main_subject,
    features_json:has_person::BOOLEAN         AS has_person,
    features_json:person_gender::VARCHAR      AS person_gender,
    features_json:person_size::VARCHAR        AS person_size,
    features_json:expression::VARCHAR         AS expression,
    features_json:location::VARCHAR           AS location,
    features_json:product_usage::VARCHAR      AS product_usage,
    features_json:color_tone::VARCHAR         AS color_tone
FROM DIRECTORY(@POST_IMAGES);

In [ ]:
%%sql -r dataframe_1_17
-- image_features を確認
SELECT
    image_file,
    photo_main_subject,
    has_person,
    person_gender,
    expression,
    location,
    product_usage,
    color_tone
FROM image_features
ORDER BY image_file;

In [ ]:
-- 画像が構造化されたので、売上と突き合わせられる
-- 「人物が写っている投稿」と「商品だけの投稿」でいいね数を比較
SELECT
    f.has_person,
    COUNT(*)                  AS post_count,
    ROUND(AVG(p.likes), 1)    AS avg_likes,
    ROUND(AVG(p.comments), 1) AS avg_comments
FROM posts p
JOIN image_features f ON p.image_path = f.image_file
GROUP BY f.has_person
ORDER BY avg_likes DESC;